In [ ]:
from datetime import datetime
semana=datetime.today().strftime('%U')
semana_atual= int(semana) + 1
print(semana_atual)

In [ ]:
import pandas as pd
ExternalGUtranCell= pd.read_csv('C:/Dumps/LTE_total/5G_ExternalNRCellCU.txt', sep='\t', usecols=['UF', 'gNB', 'CGI', 'nRPCI', 'externalNRCellCUId'],
                              low_memory=False)

ExternalGUtranCell.sample(5)

## Juntando o NRCellCU e NRCellDU para obtermos as informações de CGI e nRPCI

In [ ]:
# Criando uma variavel para utilizar a semana atual
arquivo_NRCellDU= 'C:/Camilo/' + '5G_NRCellDU'+ '_'+str(semana_atual)+'.txt'
arquivo_NRCellCU= 'C:/Camilo/' + '5G_NRCellCU'+ '_'+str(semana_atual)+'.txt'

# Abrindo o MO NRCellDU com apenas as colunas necessarias
NRCellDU= pd.read_csv(arquivo_NRCellDU, sep='\t', encoding='ISO-8859-1', usecols=['UF', 'eNB', 'nRCellDUId', 'nRPCI', 'operationalState']
                      ,low_memory=False)
# NRCellDU= NRCellDU[NRCellDU['operationalState']=='ENABLED']
# NRCellDU.drop(columns=['operationalState'],inplace=True)

# Renomeando a coluna antes de fazer o join
NRCellDU.rename(columns={'nRCellDUId': 'nRCell'},inplace=True)
NRCellDU.head()

In [ ]:
# Abrindo o MO NRCellCU com apenas as colunas necessarias 
NRCellCU= pd.read_csv(arquivo_NRCellCU, sep='\t', encoding='ISO-8859-1', usecols=['UF', 'eNB', 'nRCellCUId', 'CGI']
                      ,low_memory=False)

# Renomeando a coluna antes de fazer o join
NRCellCU.rename(columns={'nRCellCUId': 'nRCell'},inplace=True)

# Fazendo o join dos MOs de NRCell
NRCell= pd.merge(NRCellCU, NRCellDU, on=['UF', 'eNB', 'nRCell'], how='inner')

# Alterando os nomes para fazer o join com a external
NRCell.rename(columns={'eNB': 'gNB_tgt', 'nRCell':'nRCellCUId_tgt', 'CGI':'CGI_tgt', 'nRPCI':'nRPCI_nRCell', 'UF': 'UF_NRCell'},inplace=True)
NRCell.head()

In [ ]:
audit= pd.merge(ExternalGUtranCell, NRCell,left_on=['CGI'], right_on=['CGI_tgt'], how='inner')
audit= audit[audit['nRPCI']!=audit['nRPCI_nRCell']]
audit['Regional']= audit['UF'].replace(['AM', 'AP', 'MA', 'PA', 'RR', 'BA', 'SE'], ['N','N','N','N','N','BASE','BASE'])
# Movendo a coluna 'Regional' para a primeira posição
coluna = audit.pop('Regional')
audit.insert(0, 'Regional', coluna)

audit.head()

In [ ]:
# Esse bloco foi criado para obtermos o operationalState da src. pois caso ela esteja DISABLED, nao deverá constar no audit

NRCell1= NRCell[['gNB_tgt', 'operationalState']]
NRCell1= NRCell1.copy()
# NRCell1.rename(columns={'gNB_tgt':'eNB'},inplace=True)
NRCell1.drop_duplicates(inplace=True)
NRCell1= NRCell1[NRCell1['operationalState']=='ENABLED']
NRCell1.rename(columns={'operationalState': 'operationalState_src'},inplace=True)
NRCell1.head()

In [ ]:
audit1= pd.merge(audit, NRCell1, left_on='gNB', right_on='gNB_tgt', how='inner')
audit1.rename(columns={'gNB_tgt_x':'gNB_tgt', 'operationalState': 'operationalState_tgt'},inplace=True)
audit1.drop(columns='gNB_tgt_y', inplace=True)
audit1.head()

In [ ]:
audit1.to_excel('C:/Vivo Reports/5G/5G Mobilidade/Audit_NRPCI_ExternalNRCellCU.xlsx', index=None)
print('Done')